In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, 
                              confusion_matrix, 
                              roc_auc_score, 
                              roc_curve,
                              ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

# Chart style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ All libraries loaded")

 Load Cleaned Data

In [ ]:
#load cleaned data
data = pd.read_csv(r"C:\Users\user\credit-card-fraud-detection\data\fraud_cleaned.csv")

print("Shape:", data.shape)
print("\nColumns:", data.columns.tolist())
print("\nFraud Distribution:")
print(data['is_fraud'].value_counts())
print(data['is_fraud'].value_counts(normalize=True).mul(100).round(2))

Fraud vs Legitimate Count

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = data['is_fraud'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, 
             color=['steelblue', 'crimson'], edgecolor='black')
axes[0].set_title('Transaction Count: Fraud vs Legitimate')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Percentage pie
axes[1].pie(counts.values, labels=['Legitimate', 'Fraud'],
            colors=['steelblue', 'crimson'],
            autopct='%1.2f%%', startangle=90)
axes[1].set_title('Fraud vs Legitimate Percentage')

plt.suptitle('Class Imbalance Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/chart1_class_imbalance.png', dpi=150)
plt.show()

Fraud by Category

In [ ]:
fraud_by_cat = data[data['is_fraud'] == 1].groupby('category').size().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
bars = plt.bar(fraud_by_cat.index, fraud_by_cat.values, color='crimson', edgecolor='black')
plt.title('Fraud Cases by Merchant Category', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Fraud Cases')
plt.xticks(rotation=45, ha='right')

for bar, val in zip(bars, fraud_by_cat.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../notebooks/chart2_fraud_by_category.png', dpi=150)
plt.show()

Fraud by Hour

In [ ]:
hour_data = data.groupby('trans_hour')['is_fraud'].agg(['sum', 'count'])
hour_data['fraud_rate'] = hour_data['sum'] / hour_data['count'] * 100

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(hour_data.index, hour_data['sum'], color='crimson', 
        alpha=0.6, label='Fraud Cases')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Fraud Cases', color='crimson')

ax2 = ax1.twinx()
ax2.plot(hour_data.index, hour_data['fraud_rate'], 
         color='darkblue', marker='o', linewidth=2, label='Fraud Rate %')
ax2.set_ylabel('Fraud Rate %', color='darkblue')

plt.title('Fraud Cases & Rate by Hour of Day', fontsize=14, fontweight='bold')
ax1.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig('../notebooks/chart3_fraud_by_hour.png', dpi=150)
plt.show()

Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Legitimate
data[data['is_fraud'] == 0]['amt'].clip(upper=500).plot(
    kind='hist', bins=50, color='steelblue', 
    alpha=0.7, ax=axes[0], edgecolor='black')
axes[0].set_title('Legitimate Transaction Amounts')
axes[0].set_xlabel('Amount (capped at $500)')

# Fraud
data[data['is_fraud'] == 1]['amt'].clip(upper=500).plot(
    kind='hist', bins=50, color='crimson', 
    alpha=0.7, ax=axes[1], edgecolor='black')
axes[1].set_title('Fraud Transaction Amounts')
axes[1].set_xlabel('Amount (capped at $500)')

plt.suptitle('Transaction Amount Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/chart4_amount_distribution.png', dpi=150)
plt.show()

print(f"Avg Legitimate: ${data[data['is_fraud']==0]['amt'].mean():.2f}")
print(f"Avg Fraud:      ${data[data['is_fraud']==1]['amt'].mean():.2f}")

Fraud by Age Group & Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age group
age_fraud = data[data['is_fraud']==1]['age_group'].value_counts().sort_index()
axes[0].bar(age_fraud.index.astype(str), age_fraud.values, 
            color='darkorange', edgecolor='black')
axes[0].set_title('Fraud Cases by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Fraud Cases')

# Gender
gender_fraud = data[data['is_fraud']==1]['gender'].value_counts()
axes[1].bar(['Male' if g == 'M' else 'Female' for g in gender_fraud.index],
            gender_fraud.values, color=['steelblue', 'hotpink'], edgecolor='black')
axes[1].set_title('Fraud Cases by Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Fraud Cases')

plt.suptitle('Fraud by Demographics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/chart5_demographics.png', dpi=150)
plt.show()

Prepare Data for Modeling

In [ ]:
# Select features for the model
features = ['amt', 'trans_hour', 'age', 'city_pop',
            'category', 'gender', 'state', 'trans_day', 'trans_month']

target = 'is_fraud'

model_df = data[features + [target]].copy()

# Encode categorical columns
cat_cols = ['category', 'gender', 'state', 'trans_day', 'trans_month']
le = LabelEncoder()

for col in cat_cols:
    model_df[col] = le.fit_transform(model_df[col].astype(str))

print("✅ Encoding done")
print(model_df.head())

Split Data

In [ ]:
X = model_df.drop(columns=[target])
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print(f"\nFraud in train: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Fraud in test:  {y_test.sum()} ({y_test.mean()*100:.2f}%)")

Handle Class Imbalance with SMOTE

In [ ]:
print("Before SMOTE:", X_train.shape, y_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE: ", X_train_sm.shape, pd.Series(y_train_sm).value_counts().to_dict())
print("✅ Classes are now balanced")

Train Random Forest Model

In [ ]:
#train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_sm, y_train_sm)
print("✅ Model trained")

Evaluate the Model

In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, 
      target_names=['Legitimate', 'Fraud']))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Legitimate', 'Fraud'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/chart6_confusion_matrix.png', dpi=150)
plt.show()

ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score   = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, color='crimson', linewidth=2,
         label=f'Random Forest (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Fraud Detection Model', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../notebooks/chart7_roc_curve.png', dpi=150)
plt.show()

Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
bars = plt.barh(importance_df['Feature'], importance_df['Importance'],
                color='steelblue', edgecolor='black')
plt.xlabel('Importance Score')
plt.title('Feature Importance — What Drives Fraud?', 
          fontsize=13, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../notebooks/chart8_feature_importance.png', dpi=150)
plt.show()

print(importance_df)

Save the Model

In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(rf_model, '../models/fraud_rf_model.pkl')
print("✅ Model saved to models/fraud_rf_model.pkl")